# 32

Combine multiple CSV files into a single DataFrame, extracting city and state from the filename:

In [211]:
import pandas as pd
import numpy as np
import glob
import os

In [212]:
all_dfs = []

for one_filename in glob.glob('./data/*,*.csv'):
    print(f'Loading {one_filename}...')

    # Use os.path.basename to get just the filename without the directory path
    basename = os.path.basename(one_filename)

    # Extract city and state from the basename
    city, state = basename.removesuffix('.csv').split(',')

    one_df = (
        pd
        .read_csv(one_filename,
                  usecols=[0, 1, 2],
                  names=['date_time',
                         'max_temp',
                         'min_temp'],
                  header=0)
        .assign(city=city.replace('+', ' ').title(),
                state=state.upper())
    )

    all_dfs.append(one_df)

df = pd.concat(all_dfs)

Loading ./data\albany,ny.csv...
Loading ./data\boston,ma.csv...
Loading ./data\chicago,il.csv...
Loading ./data\los+angeles,ca.csv...
Loading ./data\new+york,ny.csv...
Loading ./data\san+francisco,ca.csv...
Loading ./data\springfield,il.csv...
Loading ./data\springfield,ma.csv...


Does the data for each city and state start and end at (roughly) the same time?

In [213]:
df.groupby(['state', 'city'])['date_time'].min().sort_values()

state  city         
CA     Los Angeles      2018-12-11 00:00:00
       San Francisco    2018-12-11 00:00:00
IL     Chicago          2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
MA     Boston           2018-12-11 00:00:00
       Springfield      2018-12-11 00:00:00
NY     Albany           2018-12-11 00:00:00
       New York         2018-12-11 00:00:00
Name: date_time, dtype: object

In [214]:
df.groupby(['state', 'city'])['date_time'].max().sort_values()

state  city         
CA     Los Angeles      2019-03-11 21:00:00
       San Francisco    2019-03-11 21:00:00
IL     Chicago          2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
MA     Boston           2019-03-11 21:00:00
       Springfield      2019-03-11 21:00:00
NY     Albany           2019-03-11 21:00:00
       New York         2019-03-11 21:00:00
Name: date_time, dtype: object

In [215]:
df.groupby(['state', 'city'])['date_time'].agg(['min', 'max'])

min                  max
state city                                                   
CA    Los Angeles    2018-12-11 00:00:00  2019-03-11 21:00:00
      San Francisco  2018-12-11 00:00:00  2019-03-11 21:00:00
IL    Chicago        2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
MA    Boston         2018-12-11 00:00:00  2019-03-11 21:00:00
      Springfield    2018-12-11 00:00:00  2019-03-11 21:00:00
NY    Albany         2018-12-11 00:00:00  2019-03-11 21:00:00
      New York       2018-12-11 00:00:00  2019-03-11 21:00:00

What is the lowest minimum temperature recorded for each city in the data set?

In [216]:
df.groupby(['state', 'city'])['min_temp'].min()

state  city         
CA     Los Angeles       4
       San Francisco     3
IL     Chicago         -28
       Springfield     -25
MA     Boston          -14
       Springfield     -20
NY     Albany          -19
       New York        -14
Name: min_temp, dtype: int64

What is the highest maximum temperature recorded in each state in the data set?

In [217]:
df.groupby('state')['max_temp'].max()

state
CA    23
IL    16
MA    17
NY    15
Name: max_temp, dtype: int64

Run "describe" on the minimum and maximum temperature for each state-city combination

In [218]:
df.groupby(['state', 'city'])[['max_temp', 'min_temp']].apply(pd.DataFrame.describe)

max_temp    min_temp
state city                                     
CA    Los Angeles count  728.000000  728.000000
                  mean    17.054945   10.637363
                  std      2.708640    2.705200
                  min     12.000000    4.000000
                  25%     15.000000    9.000000
...                             ...         ...
NY    New York    min    -12.000000  -14.000000
                  25%      2.000000   -4.000000
                  50%      4.000000    0.000000
                  75%      7.000000    2.000000
                  max     15.000000   12.000000

[64 rows x 2 columns]

What is the average difference in temperature (i.e., max - min) for each of the cities in our data set?

In [219]:
df.groupby(['state', 'city'])[['min_temp', 'max_temp']].apply(lambda g: np.mean(g.max() - g.min()) )

state  city         
CA     Los Angeles      12.0
       San Francisco     8.0
IL     Chicago          34.0
       Springfield      35.5
MA     Boston           26.0
       Springfield      28.5
NY     Albany           26.5
       New York         26.5
dtype: float64

# 33

Read in the scores file (`sat-scores.csv`). This time, you want the following columns: `Year`, `State.Code`, `Total.Math`, `Family Income.Less than 20k.Math`,
`Family Income.Between 20-40k.Math`, `Family Income.Between 40-60k.Math`,
`Family Income.Between 60-80k.Math`, `Family Income.Between 80-100k.Math`,
and `Family Income.More than 100k.Math`.

In [220]:
filename = './data/sat-scores.csv'

df = pd.read_csv(filename,
                usecols=['Year', 'State.Code', 'Total.Math',
                         'Family Income.Less than 20k.Math',
                         'Family Income.Between 20-40k.Math',
                         'Family Income.Between 40-60k.Math',
                         'Family Income.Between 60-80k.Math',
                         'Family Income.Between 80-100k.Math',
                         'Family Income.More than 100k.Math'])
df.head()

,Year,State.Code,Total.Math,Family Income.Between 20-40k.Math,Family Income.Between 40-60k.Math,Family Income.Between 60-80k.Math,Family Income.Between 80-100k.Math,Family Income.Less than 20k.Math,Family Income.More than 100k.Math
0,2005,AL,559,513,539,550,566,462,588
1,2005,AK,519,492,517,513,528,464,541
2,2005,AZ,530,498,520,524,534,485,554
3,2005,AR,552,513,543,553,570,489,572
4,2005,CA,522,477,506,521,535,451,566


Rename the income-related column names to something shorter:
`income<20k`, `20k<income<40k`, `40k<income<60k`, `60k<income<80k`, `80k<income<100k`, and `income>100k`.

In [221]:
df.columns = ['Year', 'State.Code', 'Total.Math',
              'income<20k',
              '20k<income<40k',
              '40k<income<60k',
              '60k<income<80k',
              '80k<income<100k',
              'income>100k',
              ]

Find the average SAT math score for each income level, grouped and then
sorted by year

In [222]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .round(3)
)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
income<20k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20k<income<40k,0.070,0.041,0.050,0.046,0.044,0.046,0.037,0.041,0.043,0.035,0.045
40k<income<60k,0.026,0.021,0.026,0.003,0.005,0.023,0.030,0.026,0.017,0.024,0.026
60k<income<80k,0.024,0.029,0.023,0.015,0.021,0.025,0.025,0.024,0.033,0.030,0.028
80k<income<100k,-0.221,-0.162,-0.161,-0.142,-0.147,-0.129,-0.150,-0.148,-0.127,-0.154,-0.174
income>100k,0.338,0.242,0.234,0.180,0.215,0.193,0.223,0.215,0.185,0.209,0.259


Find the average SAT math score for each income level, grouped and then
sorted by year

In [223]:
df.groupby('Year').mean(numeric_only=True).sort_index().round(3)

,Total.Math,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,,
2005,535.654,488.654,522.673,536.077,548.942,427.596,572.173
2006,537.481,502.923,523.769,534.904,550.462,461.019,572.519
2007,535.340,494.849,519.491,533.189,545.698,457.925,565.170
2008,535.981,523.623,547.472,549.189,557.642,478.642,564.566
2009,540.804,527.824,550.980,553.941,565.333,482.059,585.784
2010,540.843,499.275,522.000,534.235,547.627,477.039,569.275
2011,533.226,494.887,513.415,528.660,541.849,460.453,563.245
2012,533.604,492.057,512.453,525.774,538.302,458.774,557.321
2013,532.623,490.132,511.377,520.321,537.396,469.358,556.340


For each year in the data set, determine how much better each income group
did, on average, than the next-poorer group of students.

In [224]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .T
    .mean()
    .sort_values(ascending=False)
    .head()
    .round(3)
)

income>100k        0.227
20k<income<40k     0.045
60k<income<80k     0.025
40k<income<60k     0.021
80k<income<100k   -0.156
dtype: float64

Which income levels consistently (i.e., across all years) do worse than the next-poorest group?

In [225]:
change = (
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .pct_change(axis='columns')
)

change[change < 0].dropna()

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,


Calculate descriptive statistics for all the changes in income brackets

In [226]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()
change.T.describe().round(3)

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
count,0.0,11.000,11.000,11.000,11.000,11.000
mean,NaN,0.045,0.021,0.025,-0.156,0.227
std,NaN,0.009,0.009,0.005,0.026,0.044
min,NaN,0.035,0.003,0.015,-0.221,0.180
25%,NaN,0.041,0.019,0.024,-0.162,0.201
50%,NaN,0.044,0.024,0.025,-0.150,0.215
75%,NaN,0.046,0.026,0.029,-0.144,0.238
max,NaN,0.070,0.030,0.033,-0.127,0.338


Which five states have the greatest gap in SAT math scores between the richest
and poorest students?

In [227]:
df['rich_poor_diff'] = df['income>100k'] - df['income<20k']
df.groupby('State.Code')['rich_poor_diff'].mean().sort_values(ascending=False).head().round(2)

State.Code
DC    182.00
MD     95.18
CT     95.00
DE     91.55
NJ     86.00
Name: rich_poor_diff, dtype: float64

Perform the same analysis on verbal SAT scores

In [228]:
df = pd.read_csv(filename,
                usecols=['Year', 'State.Code', 'Total.Verbal',
                         'Family Income.Less than 20k.Verbal',
                         'Family Income.Between 20-40k.Verbal',
                         'Family Income.Between 40-60k.Verbal',
                         'Family Income.Between 60-80k.Verbal',
                         'Family Income.Between 80-100k.Verbal',
                         'Family Income.More than 100k.Verbal'])

df.columns = ['Year', 'State.Code', 'Total.Verbal',
                      'income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k',
                      ]

df.head()

,Year,State.Code,Total.Verbal,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
0,2005,AL,567,527,551,564,577,474,590
1,2005,AK,523,500,522,519,534,467,544
2,2005,AZ,526,495,518,523,533,474,546
3,2005,AR,563,526,555,570,580,486,589
4,2005,CA,504,458,494,511,525,421,551


In [229]:
(
    df
    .groupby('Year')
    [['income<20k',
      '20k<income<40k',
      '40k<income<60k',
      '60k<income<80k',
      '80k<income<100k',
      'income>100k']]
    .mean()
    .T
    .pct_change()
    .round(3)
)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
income<20k,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20k<income<40k,0.048,0.043,0.052,0.040,0.038,0.046,0.043,0.044,0.047,0.032,0.045
40k<income<60k,0.022,0.018,0.021,0.003,0.006,0.023,0.029,0.026,0.018,0.035,0.027
60k<income<80k,0.023,0.022,0.018,0.014,0.023,0.025,0.017,0.015,0.026,0.018,0.021
80k<income<100k,-0.165,-0.170,-0.167,-0.141,-0.152,-0.143,-0.160,-0.156,-0.139,-0.163,-0.184
income>100k,0.240,0.245,0.238,0.174,0.209,0.207,0.238,0.222,0.199,0.214,0.269


In [230]:
df.groupby('Year').mean(numeric_only=True).sort_index().round(2)

,Total.Verbal,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
Year,,,,,,,
2005,534.27,501.10,524.96,536.25,548.40,457.83,567.52
2006,531.73,502.15,523.96,533.33,544.87,452.46,563.27
2007,531.53,496.58,522.43,533.17,542.81,452.40,559.98
2008,530.72,522.92,543.70,545.28,552.91,474.83,557.47
2009,535.14,526.84,547.00,550.02,562.57,476.92,576.61
2010,535.86,497.65,520.57,532.61,545.82,467.65,564.39
2011,528.81,493.21,514.17,529.25,538.11,451.91,559.38
2012,527.36,491.11,512.89,526.23,534.32,450.85,550.83
2013,528.32,490.06,513.02,522.42,536.08,461.64,553.49


In [231]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()
change.T.describe().round(3)

,income<20k,20k<income<40k,40k<income<60k,60k<income<80k,80k<income<100k,income>100k
count,0.0,11.000,11.000,11.000,11.000,11.000
mean,NaN,0.043,0.021,0.020,-0.158,0.223
std,NaN,0.005,0.010,0.004,0.014,0.026
min,NaN,0.032,0.003,0.014,-0.184,0.174
25%,NaN,0.041,0.018,0.017,-0.166,0.208
50%,NaN,0.044,0.022,0.021,-0.160,0.222
75%,NaN,0.046,0.027,0.023,-0.148,0.239
max,NaN,0.052,0.035,0.026,-0.139,0.269


In [232]:
change = df.groupby('Year')[['income<20k',
                      '20k<income<40k',
                      '40k<income<60k',
                      '60k<income<80k',
                      '80k<income<100k',
                      'income>100k']].mean().T.pct_change()

change[change <= 0].dropna().round(3)

Year,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015
80k<income<100k,-0.165,-0.17,-0.167,-0.141,-0.152,-0.143,-0.16,-0.156,-0.139,-0.163,-0.184


In [233]:
df['rich_poor_diff'] = df['income>100k'] - df['income<20k']
df.groupby('State.Code')['rich_poor_diff'].mean().sort_values(ascending=False).head().round(2)

State.Code
DC    182.45
MD     89.09
DE     86.45
CT     84.55
CA     84.09
Name: rich_poor_diff, dtype: float64

# 34

In [234]:
all_dfs = []

for one_filename in glob.glob('./data/*,*.csv'):
    print(f'Loading {one_filename}...')

    # Use os.path.basename to get just the filename without the directory path
    basename = os.path.basename(one_filename)

    # Extract city and state from the basename
    city, state = basename.removesuffix('.csv').split(',')

    one_df = (
        pd
        .read_csv(one_filename,
                  usecols=[1, 2, 19],
                  names=['max_temp',
                         'min_temp',
                         'precipMM'],
                  header=0)
        .assign(city=city.replace('+', ' ').title(),
                state=state.upper())
    )

    all_dfs.append(one_df)

df = pd.concat(all_dfs)

Loading ./data\albany,ny.csv...
Loading ./data\boston,ma.csv...
Loading ./data\chicago,il.csv...
Loading ./data\los+angeles,ca.csv...
Loading ./data\new+york,ny.csv...
Loading ./data\san+francisco,ca.csv...
Loading ./data\springfield,il.csv...
Loading ./data\springfield,ma.csv...


In [235]:
df.head()

,max_temp,min_temp,precipMM,city,state
0,-2,-8,0.0,Albany,NY
1,-2,-8,0.0,Albany,NY
2,-2,-8,0.0,Albany,NY
3,-2,-8,0.0,Albany,NY
4,-2,-8,0.0,Albany,NY


Determine which cities had, on at least three occasions, precipitation of 15 mm or more.

In [236]:
df[df['precipMM'] >= 15].groupby(['state', 'city']).size().loc[lambda x: x >= 3]

state  city       
CA     Los Angeles    7
MA     Boston         3
NY     New York       4
dtype: int64

Find cities that had at least three measurements of 10 mm of precipitation or more when the temperature was at or below 0° Celsius.

In [237]:
# Either max or min temp is <= 0
df[(df['precipMM'] >= 10) & ((df['max_temp'] <= 0) | (df['min_temp'] <= 0))].groupby(['state', 'city']).size().loc[lambda x: x >= 3]

state  city    
MA     Boston      6
NY     Albany      3
       New York    6
dtype: int64

In [238]:
# Both max and min temp are <= 0
df[(df['precipMM'] >= 10) & (df['max_temp'] <= 0) & (df['min_temp'] <= 0)].groupby(['state', 'city']).size().loc[lambda x: x >= 3]

state  city  
NY     Albany    3
dtype: int64

For each precipitation measurement, calculate the proportion of that city’s total precipitation.

In [239]:
def proportion_total_city_precip(s):
    """
    Calculate the proportion of total precipitation for each city.
    """
    return s / s.sum()

df['precip_pct'] = (
    df
    .groupby('city')['precipMM']
    .transform(proportion_total_city_precip)
) * 100  # Convert to percentage

df.groupby(['city', 'state'])['precip_pct'].max().round(2)

city           state
Albany         NY       2.92
Boston         MA       4.83
Chicago        IL       5.73
Los Angeles    CA       5.92
New York       NY       5.51
San Francisco  CA       5.65
Springfield    IL       3.10
               MA       2.35
Name: precip_pct, dtype: float64

# 35

Create a data frame, `oecd_df`, from `oecd_locations.csv`, containing a subset of all OECD countries.

The resulting data set should have a single column called `country`. The index should be based on the country’s abbreviation.

In [240]:
oecd_df = pd.read_csv('./data/oecd_locations.csv',
                      header=None,
                      names=['abbrev', 'country'],
                      index_col='abbrev')
oecd_df.head()

,country
abbrev,
AUS,Australia
AUT,Austria
BEL,Belgium
CAN,Canada
DNK,Denmark


Create a second data frame, `oecd_tourism_df`, from `oecd_tourism.csv`. You’re only interested in four columns: `LOCATION` (which will serve as the index), `TIME`, `SUBJECT`, and `Value`.

You’re also only interested in rows where `SUBJECT` has the value `INT-EXP`. Once you’ve kept only the rows with `INT-EXP`, you can remove the `SUBJECT` column

In [241]:
oecd_tourism_df = (pd.read_csv('./data/oecd_tourism.csv',
                               usecols=['LOCATION', 'TIME', 'Value', 'SUBJECT'],
                               index_col='LOCATION')
                   .loc[lambda df_: df_['SUBJECT'] == 'INT-EXP']
                   .drop('SUBJECT', axis='columns')
                   )
oecd_tourism_df.head()

,TIME,Value
LOCATION,,
AUS,2008,27620.0
AUS,2009,25629.6
AUS,2010,31916.5
AUS,2011,39381.5
AUS,2012,41632.8


Create a new series, `tourism_spending`, in which the index reflects the country names (i.e., not abbreviations) and the value contains the average tourism
spending for that country.

In [242]:
tourism_spending = (
    oecd_df
    .join(oecd_tourism_df)
    .groupby('country')['Value'].mean().round(2)
)

tourism_spending.head(10)

country
Australia    36727.97
Austria      11934.56
Belgium      20859.88
Brazil       21564.35
Canada       40984.63
Denmark      11326.17
Finland       5877.08
France       51394.27
Germany      96615.08
Hungary       2918.39
Name: Value, dtype: float64

Create a third data frame, `wine_df`, based on `winemag-150k-reviews.csv`. You only need two columns: `country` and `points`

In [243]:
wine_df = pd.read_csv('./data/winemag-150k-reviews.csv',
                        usecols=['country', 'points']
                   )
wine_df.head()

,country,points
0,US,96
1,Spain,96
2,US,96
3,US,96
4,France,95


Get the mean wine score for each country, across all wine reviews, sorted in descending order.

In [244]:
country_points = (wine_df
                  .groupby('country')['points']
                  .mean()
                  .round(2)
                  .sort_values(ascending=False)
                  )
country_points.head()

country
England    92.89
Austria    89.28
France     88.93
Germany    88.63
Italy      88.41
Name: points, dtype: float64

Perform a standard join between the average wine scores per country and the average tourism spending per country.

The left data frame (in this case, `country_points.to_frame()`) dictates the index of the data frame that results from the join - hence why it is known as a *left join*.

In a left join, columns from the right frame are missing values (and thus have `NaN`) wherever there was no corresponding row for the left’s index.

In [245]:
country_points.to_frame().join(tourism_spending)

,points,Value
country,,
England,92.89,NaN
Austria,89.28,11934.56
France,88.93,51394.27
Germany,88.63,96615.08
Italy,88.41,34148.91
Canada,88.24,40984.63
Slovenia,88.23,NaN
Morocco,88.17,NaN
Turkey,88.10,NaN


The `join` method defaults to a left join. The `how` parameter can be used to specify the type of join you want to perform.

Which, in this case, doesn't really improve the end result.

In [246]:
country_points.to_frame().join(tourism_spending, how='right')

,points,Value
country,,
Australia,87.89,36727.97
Austria,89.28,11934.56
Belgium,NaN,20859.88
Brazil,83.24,21564.35
Canada,88.24,40984.63
Denmark,NaN,11326.17
Finland,NaN,5877.08
France,88.93,51394.27
Germany,88.63,96615.08


Perform an outer join between the average wine scores per country and the average tourism spending per country.

In an outer join, the output data frame index is a combination of the indices from both left and right data frames. Wherever there is no corresponding row for the left’s index, the right’s values will be `NaN`, and vice versa.

In [247]:
country_points.to_frame().join(tourism_spending, how='outer')

,points,Value
country,,
Albania,88.00,NaN
Argentina,86.00,NaN
Australia,87.89,36727.97
Austria,89.28,11934.56
Belgium,NaN,20859.88
Bosnia and Herzegovina,84.75,NaN
Brazil,83.24,21564.35
Bulgaria,85.47,NaN
Canada,88.24,40984.63


Find the correlation between average wine score and average tourism spending.

With a correlation of 0.3, there is a weak positive correlation between the two variables.

Remember that correlation does not imply causation!

In [248]:
country_points.to_frame().join(tourism_spending, how='outer').corr().round(3)

,points,Value
points,1.000,0.288
Value,0.288,1.000


Read in the three data frames, but without setting an index. Ensure that the column names in `oecd_tourism_df` are `abbrev`, `TIME`, and `Value`, and that the dtype of the `Value` column is `np.int64`.

In [249]:
oecd_df = pd.read_csv('./data/oecd_locations.csv',
                      header=None,
                      names=['abbrev', 'country'])

oecd_tourism_df = pd.read_csv('./data/oecd_tourism.csv',
                              usecols=[0, 5,6],
                              header=0,
                              names=['abbrev', 'TIME', 'Value'])

wine_df = pd.read_csv('./data/winemag-150k-reviews.csv',
                      usecols=['country', 'points'])

Perform the same joins as before, but using `merge`, rather than `join`.

In [254]:
tourism_spending = (oecd_df
                    .merge(oecd_tourism_df, on='abbrev')
                    .groupby('country')['Value'].mean().round(2)
                   )
print(tourism_spending)

country
Australia          37634.43
Austria            16673.89
Belgium            16525.24
Brazil             13942.91
Canada             32593.61
Denmark            10362.56
Finland             5288.66
France             58228.80
Germany            75011.82
Hungary             5108.87
Israel              6634.45
Italy              39539.56
Japan              28606.89
Korea              21677.13
United Kingdom     63507.16
United States     171847.08
Name: Value, dtype: float64


In [255]:
country_points = (
    wine_df
    .groupby('country')['points'].mean().round(2)
)

print(country_points)

country
Albania                   88.00
Argentina                 86.00
Australia                 87.89
Austria                   89.28
Bosnia and Herzegovina    84.75
Brazil                    83.24
Bulgaria                  85.47
Canada                    88.24
Chile                     86.30
China                     82.00
Croatia                   86.28
Cyprus                    85.87
Czech Republic            85.83
Egypt                     83.67
England                   92.89
France                    88.93
Georgia                   85.51
Germany                   88.63
Greece                    86.12
Hungary                   87.33
India                     87.62
Israel                    87.18
Italy                     88.41
Japan                     85.00
Lebanon                   85.70
Lithuania                 84.25
Luxembourg                87.00
Macedonia                 84.81
Mexico                    84.76
Moldova                   84.72
Montenegro                82.00


In [252]:
(
    country_points.to_frame()
    .merge(tourism_spending, on='country')
)

,points,Value
country,,
Australia,87.89,37634.43
Austria,89.28,16673.89
Brazil,83.24,13942.91
Canada,88.24,32593.61
France,88.93,58228.80
Germany,88.63,75011.82
Hungary,87.33,5108.87
Israel,87.18,6634.45
Italy,88.41,39539.56


In [253]:
(
    country_points.to_frame()
    .merge(tourism_spending, on='country', how='outer')
)

,points,Value
country,,
Albania,88.00,NaN
Argentina,86.00,NaN
Australia,87.89,37634.43
Austria,89.28,16673.89
Belgium,NaN,16525.24
Bosnia and Herzegovina,84.75,NaN
Brazil,83.24,13942.91
Bulgaria,85.47,NaN
Canada,88.24,32593.61
